# Fase 2 - Procesamiento distribuido con PySpark

Terminal Puerto Coquimbo, sitio 3. El notebook corre en el contenedor `jupyter`
(el driver) contra el cluster `spark://spark-master:7077`, y lee los JSONL que
dejo la Fase 1.

| Hito | Contenido |
|---|---|
| 1 | Configuracion de la sesion e ingesta con esquema explicito |
| 2 | Perfilado y diagnostico de calidad |
| 3 | Limpieza, normalizacion y filtrado |
| 4 | Diagnostico y optimizacion del shuffle |
| 5 | Escritura en Parquet (`salida_fase2/`) |
| 6 | Metricas y conclusiones |

## Hito 1 - Configuracion e ingesta con esquema explicito

### 1.1 Parametros

La Fase 1 (`src/orquestador.py`) escribe una subcarpeta por configuracion de la
campana de benchmarking (`data/raw/w01`, `w02`, `w04`, `w08`, `w16`), y **cada
una es el dataset completo de 10 M de eventos**. Leer `data/raw/*/*.jsonl`
mezclaria cinco corridas (~50 M de filas con series repetidas por sensor), y
`data/raw/*.jsonl` no encuentra nada. Por eso se lee una sola carpeta,
elegida con `CONFIG_FASE1`.

La ruta es la del contenedor: `docker-compose.yml` monta el proyecto en
`/opt/workspace` en los tres servicios, asi que driver y executors ven la misma
ruta y cada executor abre sus archivos por su cuenta.

In [ ]:
import glob
import os
import time

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType, IntegerType, StringType, StructField, StructType,
)

RUTA_PROYECTO = "/opt/workspace"

# subcarpeta de la fase 1 que se procesa (w01, w02, w04, w08 o w16)
CONFIG_FASE1 = "w16"

RUTA_RAW = f"{RUTA_PROYECTO}/data/raw/{CONFIG_FASE1}"
PATRON_RAW = f"{RUTA_RAW}/*.jsonl"
RUTA_SALIDA = f"{RUTA_PROYECTO}/salida_fase2"   # en .gitignore, se usa en el hito 5

# valores esperados del dominio, salen de src/simulador.py (_inventario)
EQUIPOS_ESPERADOS = 69
PUNTOS_ESPERADOS = 185
METRICAS_ESPERADAS = {"vibration", "current", "weight", "temperature", "humidity"}
EVENTOS_ESPERADOS = 10_000_000   # src/main.py, TOTAL_EVENTOS

### 1.2 Comprobacion previa de los archivos

Se revisa desde el driver antes de levantar la sesion: si la carpeta no existe,
Spark no falla al definir la lectura sino recien en la primera accion, y el
error llega envuelto en un stack de Java.

In [ ]:
archivos = sorted(glob.glob(PATRON_RAW))
if not archivos:
    raise FileNotFoundError(
        f"No hay .jsonl en {RUTA_RAW}. Revisa CONFIG_FASE1 o genera los datos "
        "con la fase 1 (python src/main.py)."
    )

total_bytes = sum(os.path.getsize(a) for a in archivos)
print(f"Archivos : {len(archivos)}")
print(f"Volumen  : {total_bytes / 1024**3:.2f} GiB")
for a in archivos[:5]:
    print(f"  {os.path.basename(a):<16} {os.path.getsize(a) / 1024**2:8.1f} MiB")
if len(archivos) > 5:
    print(f"  ... y {len(archivos) - 5} mas")

### 1.3 Sesion Spark

El master, `spark.driver.host`, la zona horaria UTC y AQE ya vienen de
`docker/spark-defaults.conf`, que la imagen copia a `/opt/spark/conf`. Aqui solo
se fija lo que depende del `.env`: la memoria del executor (tiene que ser
<= `SPARK_WORKER_MEMORY` o el worker no lanza executors).

`spark.sql.shuffle.partitions` queda en el default (200) a proposito; su ajuste
es materia del hito 4, con el plan de ejecucion a la vista.

In [ ]:
spark = (
    SparkSession.builder
    .appName("fase2-puerto-coquimbo")
    .config("spark.executor.memory", os.environ.get("SPARK_EXECUTOR_MEMORY", "1g"))
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel("WARN")

# los executors tardan unos segundos en registrarse
esperados = int(os.environ.get("SPARK_WORKERS", "1"))
for _ in range(30):
    ejecutores = sc._jsc.sc().getExecutorMemoryStatus().size() - 1   # -1 = el driver
    if ejecutores >= esperados:
        break
    time.sleep(1)

print(f"Spark            : {spark.version}")
print(f"Master           : {sc.master}")
print(f"Executors vivos  : {ejecutores} (esperados {esperados})")
print(f"Paralelismo      : {sc.defaultParallelism}")
print(f"Zona horaria     : {spark.conf.get('spark.sql.session.timeZone')}")
print(f"AQE              : {spark.conf.get('spark.sql.adaptive.enabled')}")
print(f"maxPartitionBytes: {spark.conf.get('spark.sql.files.maxPartitionBytes')}")
if ejecutores == 0:
    raise RuntimeError("Sin executors: revisa SPARK_EXECUTOR_MEMORY <= SPARK_WORKER_MEMORY")

### 1.4 Esquema explicito

El contrato del evento esta en `src/esquema.py`:

```
{"timestamp":"2026-01-01T00:00:00.000Z","sensor_id":"FAJA_S3_C1","metric":"vibration","value":2.31,"unit":"mm/s","worker_id":0}
```

Por que esquema explicito y no `inferSchema`:

- **Una pasada menos sobre ~1,4 GB.** Sin esquema, Spark recorre todo el JSON
  solo para deducir los tipos antes de la lectura real.
- **Tipos estables.** La inferencia podria dar `value` como `long` si en un
  archivo todos los valores fueran enteros.

Decisiones de tipos:

- `timestamp` entra como **`StringType`**. Si se declarara `TimestampType`, en
  modo `PERMISSIVE` un formato invalido se convertiria en `null` sin dejar
  rastro, y no se podria distinguir "faltaba" de "venia mal". El parseo se hace
  explicito en el hito 3, despues de perfilar el texto crudo en el hito 2.
- `worker_id` como `IntegerType`: son ids de proceso de la fase 1 (0 a 15).
- `_corrupt_record` guarda la linea completa cuando no es JSON valido o no
  calza con el esquema. Es la materia prima del diagnostico de calidad.

In [ ]:
ESQUEMA_EVENTO = StructType([
    StructField("timestamp",       StringType(),  True),
    StructField("sensor_id",       StringType(),  True),
    StructField("metric",          StringType(),  True),
    StructField("value",           DoubleType(),  True),
    StructField("unit",            StringType(),  True),
    StructField("worker_id",       IntegerType(), True),
    StructField("_corrupt_record", StringType(),  True),
])

### 1.5 Lectura

- `mode=PERMISSIVE` + `columnNameOfCorruptRecord`: no se descarta ninguna linea
  en silencio (`DROPMALFORMED` lo haria) ni se cae el job entero por una sola
  linea mala (`FAILFAST`).
- `multiLine` queda en `false` (el default): un objeto por linea es lo que hace
  divisible el archivo, y cada archivo se reparte en bloques de
  `maxPartitionBytes` entre los executors.

La lectura es perezosa: aqui no se mueve ningun dato todavia.

In [ ]:
df_raw = (
    spark.read
    .schema(ESQUEMA_EVENTO)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .json(PATRON_RAW)
)

df_raw.printSchema()
print(f"Particiones de lectura: {df_raw.rdd.getNumPartitions()}")

### 1.6 Verificacion de la ingesta

Una muestra para confirmar que los campos llegan poblados con el esquema
declarado, y el conteo total, que es la primera accion que recorre todo el
dataset.

El `cache()` no es decorativo: Spark no permite consultas sobre JSON crudo que
solo referencien `_corrupt_record`, y el hito 2 lo va a necesitar. Ademas el
perfilado del hito 2 hace varias pasadas, y con el cache se lee el disco una
sola vez. Se verifica en la pestana *Storage* de la UI (puerto 4040) cuanto
quedo en memoria; si no cabe entero, Spark recalcula las particiones que faltan.

In [ ]:
df_raw.show(5, truncate=False)

In [ ]:
df_raw = df_raw.cache()

t0 = time.perf_counter()
total = df_raw.count()
t_lectura = time.perf_counter() - t0

print(f"Registros leidos : {total:,}")
print(f"Esperados        : {EVENTOS_ESPERADOS:,}")
print(f"Tiempo (lectura + cache): {t_lectura:.1f} s")
if total != EVENTOS_ESPERADOS:
    print("AVISO: el conteo no coincide con TOTAL_EVENTOS de la fase 1")